In [2]:
import sqlite3                                 
import pandas as pd    

import numpy as np
from scipy import stats
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     
    return pd.read_sql(sql, con)

In [3]:
def pval(z, tail='two'):                       # p from a z value
    if tail == 'greater':                      # H1: bigger
        return stats.norm.sf(z)                # area to the right
    if tail == 'less':                         # H1: smaller
        return stats.norm.cdf(z)               # area to the left
    return 2 * stats.norm.sf(abs(z))           # both sides
 
def f_test(x, y):                              # two variances
    f = np.var(x, ddof=1) / np.var(y, ddof=1)  # ratio of spreads
    d1, d2 = len(x) - 1, len(y) - 1            # degrees of freedom
    lo = stats.f.cdf(f, d1, d2)                # area to the left
    p = 2 * min(lo, stats.f.sf(f, d1, d2))     # two-tailed
    print(f'F = {f:.4f}  p = {p:.4g}')         # the result
 
def z_mean2(x, y, tail='two'):                 # two means, large
    se = np.sqrt(np.var(x, ddof=1) / len(x)    # standard error,
                 + np.var(y, ddof=1) / len(y)) # each group its own
    z = (np.mean(x) - np.mean(y)) / se         # gap, in se units
    print(f'z = {z:.4f}  p = {pval(z, tail):.4g}')  # the result


    def z_prop2(k1, n1, k2, n2, tail='two'):       # two percentages
        pool = (k1 + k2) / (n1 + n2)               # both groups as one
        se = np.sqrt(pool * (1 - pool) * (1/n1 + 1/n2))  # spread if H0
        z = (k1/n1 - k2/n2) / se                   # gap, in se units
        print(f'z = {z:.4f}  p = {pval(z, tail):.4g}')  # the result

In [4]:
res=q(""" 
SELECT c.ab_group AS g, u.customer_id,
     AVG(u.data_mb) AS m
FROM usage_daily u
JOIN customer_features c ON c.customer_id = u.customer_id
WHERE u.usage_date < '2025-05-01'
GROUP BY u.customer_id;
""")

In [5]:
a=res[res.g=='A'].m
b=res[res.g=='B'].m


f_test(a, b)  # check if the two groups have the same variance


# pvalue,0.05

z_mean2(a,b)

F = 1.1108  p = 3.673e-08
z = 0.5191  p = 0.6037


In [6]:
cf=q("""
SELECT * FROM customer_features;

""")
A,B=cf[cf.ab_group== 'A'],cf[cf.ab_group== 'B']

for col in ['stated_arpu','tenure_days','complaint_count']:
    print(col)
    print(B[col],A[col])

stated_arpu
0        374
2        374
3        257
8        257
14       257
        ... 
21993    320
21994    257
21996    257
21997    428
21998    374
Name: stated_arpu, Length: 11004, dtype: int64 1         257
4         224
5         320
6         428
7         148
         ... 
21989    1499
21990     257
21992     428
21995     257
21999     257
Name: stated_arpu, Length: 10996, dtype: int64
tenure_days
0        1030
2        1074
3        1273
8         501
14       1075
         ... 
21993     635
21994     153
21996     925
21997     410
21998     491
Name: tenure_days, Length: 11004, dtype: int64 1         506
4         198
5         456
6         761
7         224
         ... 
21989     614
21990     850
21992    1067
21995    1031
21999    1053
Name: tenure_days, Length: 10996, dtype: int64
complaint_count
0        3
2        4
3        0
8        0
14       4
        ..
21993    1
21994    4
21996    0
21997    0
21998    1
Name: complaint_count, Length: 11004, dtype: i

In [7]:
# Claim :the offer raised revenue
# H₀ :both groups paid the same on average
# H₁ :group B paid more
# Tail : one-tailed — the company said so in advance
Cev = q("""
SELECT c.ab_group AS g,
    COALESCE(SUM(r.amount_inr), 0) AS s
FROM customer_features c
LEFT JOIN recharges r ON r.customer_id=c.customer_id
    AND r.recharge_date >= '2025-01-01'
GROUP BY c.customer_id;
""")